In [ ]:
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output, Math

# ==============================================================================
# PROBLEM 9: Fully Symbolic Z-Transform via SymPy Definition Sum & Visualization
# Signal: x[n] = alpha^n u[n] + beta^n u[n] + gamma^n u[-n-1]
# ==============================================================================

explanation_text = """
<div style="background-color: #f8f9fa; padding: 10px; border-radius: 5px; border: 1px solid #dee2e6; font-size: 13px;">
<b>Solution Overview (Fully Symbolic Calculation via SymPy)</b><br>
* <b>Signal:</b> Two-sided signal x[n] = α<sup>n</sup>u[n] + β<sup>n</sup>u[n] + γ<sup>n</sup>u[-n-1] with |α| < |beta| < |γ|<br>
* <b>Z-Transform Definition:</b> X(z) = Σ x[n] z<sup>-n</sup> computed explicitly via SymPy summation.<br>
* <b>ROC (Region of Convergence):</b> Annular region |β| < |z| < |γ|.<br>
* <b>Note:</b> Use the sliders below to dynamically change parameters α, β, γ and observe the exact symbolic expansion, poles/zeros, and ROC ring.
</div>
"""
display(widgets.HTML(explanation_text))

out = widgets.Output()

# Define symbolic variables
n_sym = sp.Symbol('n', integer=True)
z_sym = sp.Symbol('z', complex=True)
alpha_sym = sp.Symbol('alpha', positive=True, real=True)
beta_sym = sp.Symbol('beta', positive=True, real=True)
gamma_sym = sp.Symbol('gamma', positive=True, real=True)

# 1. Symbolic Z-transform calculation divided into the three signal components
term1 = alpha_sym**n_sym
term2 = beta_sym**n_sym
term3 = gamma_sym**n_sym

sum1 = sp.summation(term1 * z_sym**(-n_sym), (n_sym, 0, sp.oo))
sum2 = sp.summation(term2 * z_sym**(-n_sym), (n_sym, 0, sp.oo))
sum3 = sp.summation(term3 * z_sym**(-n_sym), (n_sym, -sp.oo, -1))

X_z_sym = sp.simplify(sum1 + sum2 + sum3)
display(Math(f"X(z) = {sp.latex(X_z_sym)}"))

def plot_problem_9(alpha_val, beta_val, gamma_val):
    with out:
        clear_output(wait=True)
        
        fig, (ax_pz, ax_time) = plt.subplots(1, 2, figsize=(16, 5), gridspec_kw={'width_ratios': [1, 2]})
        plt.subplots_adjust(wspace=0.25)

        # --- 1. Pole-Zero Map & ROC ---
        ax_pz.set_aspect('equal')
        ax_pz.set_xlim(-2.5, 2.5)
        ax_pz.set_ylim(-2.5, 2.5)
        ax_pz.axhline(0, color='black', linewidth=1)
        ax_pz.axvline(0, color='black', linewidth=1)
        ax_pz.grid(True, linestyle=':', alpha=0.7)

        inner_radius = beta_val
        outer_radius = gamma_val

        # ROC: Ring-shaped region |beta| < |z| < |gamma|
        x_vals = np.linspace(-3.0, 3.0, 400)
        y_vals = np.linspace(-3.0, 3.0, 400)
        X, Y = np.meshgrid(x_vals, y_vals)
        Z_dist = np.sqrt(X**2 + Y**2)
        roc_mask = (Z_dist > inner_radius) & (Z_dist < outer_radius)

        ax_pz.imshow(roc_mask, extent=(-3.0, 3.0, -3.0, 3.0), origin='lower', cmap='Greens', alpha=0.25, zorder=0)

        theta = np.linspace(0, 2*np.pi, 200)
        ax_pz.plot(inner_radius * np.cos(theta), inner_radius * np.sin(theta), 'g:', linewidth=2, label=f'Inner ROC (|z| = |β|)')
        ax_pz.plot(outer_radius * np.cos(theta), outer_radius * np.sin(theta), 'g:', linewidth=2, label=f'Outer ROC (|z| = |γ|)')
        ax_pz.plot(np.cos(theta), np.sin(theta), 'k--', alpha=0.5)

        # Poles calculation (alpha, beta, gamma)
        poles_x = [alpha_val, beta_val, gamma_val]
        poles_y = [0, 0, 0]
        ax_pz.scatter(poles_x, poles_y, s=140, color='purple', marker='x', linewidths=3)

        # Zeros calculation from numerator roots
        coeffs = [1, -2*gamma_val, -(alpha_val*beta_val - beta_val*gamma_val - alpha_val*gamma_val)]
        roots = np.roots(coeffs)
        zeros_x = np.concatenate(([0], roots.real))
        zeros_y = np.concatenate(([0], roots.imag))
        ax_pz.scatter(zeros_x, zeros_y, s=120, facecolors='none', edgecolors='b', linewidths=2, marker='o')

        ax_pz.set_title(f'Pole-Zero Map & ROC ({inner_radius:.2f} < |z| < {outer_radius:.2f})', fontsize=10, fontweight='bold')
        ax_pz.set_xlabel('Real Part', fontsize=9)
        ax_pz.set_ylabel('Imaginary Part', fontsize=9)

        # Legend handles in a single straight line
        unit_circle_handle = plt.Line2D([0], [0], color='k', linestyle='--', alpha=0.5, label='Unit Circle')
        pole_handle = plt.Line2D([0], [0], marker='x', color='purple', markersize=8, markeredgewidth=3, linestyle='None', label='Poles (α, β, γ)')
        zero_handle = plt.Line2D([0], [0], marker='o', markerfacecolor='none', markeredgecolor='b', markersize=8, markeredgewidth=2, linestyle='None', label='Zeros (z=0, roots)')
        
        ax_pz.legend(handles=[unit_circle_handle, pole_handle, zero_handle], loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=3, fontsize=8)

        # --- 2. Time Domain Plot ---
        n_vec = np.arange(-10, 15)
        x_n_vals = (alpha_val**n_vec) * (n_vec >= 0) + (beta_val**n_vec) * (n_vec >= 0) + (gamma_val**n_vec) * (n_vec < 0)

        ax_time.stem(n_vec, x_n_vals, linefmt='r-', markerfmt='ro', basefmt='k-')
        ax_time.set_title('Temporal Evolution: x[n] = α^n u[n] + β^n u[n] + γ^n u[-n-1]', fontsize=10, fontweight='bold')
        ax_time.set_xlabel('Time index n', fontsize=9)
        ax_time.set_ylabel('x[n]', fontsize=9)
        ax_time.set_xlim(-11, 15)
        
        max_abs_val = np.max(np.abs(x_n_vals))
        y_limit = max(1.2, min(max_abs_val * 1.25, 20.0))
        ax_time.set_ylim(-0.1, y_limit)
        ax_time.grid(True, linestyle=':', alpha=0.7)

        plt.show()

# Widgets Sliders for parameters respecting |alpha| < |beta| < |gamma|
alpha_slider = widgets.FloatSlider(value=0.3, min=0.1, max=0.5, step=0.01, description='Alpha:', style={'description_width': 'initial'})
beta_slider = widgets.FloatSlider(value=0.7, min=0.55, max=0.85, step=0.01, description='Beta:', style={'description_width': 'initial'})
gamma_slider = widgets.FloatSlider(value=1.2, min=0.9, max=1.8, step=0.01, description='Gamma:', style={'description_width': 'initial'})

plot_problem_9(alpha_slider.value, beta_slider.value, gamma_slider.value)

interactive_plot = widgets.interactive(plot_problem_9, alpha_val=alpha_slider, beta_val=beta_slider, gamma_val=gamma_slider)
display(widgets.VBox([interactive_plot, out]))